In [1]:
#Load Libraries

In [2]:
import torch
import numpy as np
import lab07_starter as S

In [3]:
if __name__ == "__main__":
    Xtr, ytr, Xte, yte = S.load_task()
    net = S.train_feature_net(Xtr, ytr)
    fte = S.get_features(net, Xte)

    # 1. Six sets of 359 images evaluation
    copier = Xtr[:359] + torch.randn(359, 64) * 0.02
    mean_noise = Xtr.mean(dim=0, keepdim=True).repeat(359, 1) + torch.randn(359, 64) * 0.1
    
    sets = {
        "Real Test": Xte,
        "Copier": copier.clamp(0, 1),
        "Mean + Noise": mean_noise.clamp(0, 1),
    }
    
    for name, imgs in sets.items():
        scores = S.score_samples(imgs, fte, net, Xtr)
        print(f"Set: {name} -> {scores}")

    # 2. Subset sample size effect on Fréchet Distance
    subset_sizes = [30, 60, 120, 359, 1000]
    for size in subset_sizes:
        if size > len(Xtr): continue
        fds = []
        for _ in range(10):
            idx = torch.randperm(len(Xtr))[:size]
            sub_feats = S.get_features(net, Xtr[idx])
            fds.append(S.frechet_distance(fte, sub_feats))
        print(f"Subset size {size}: Mean FD = {np.mean(fds):.2f} (std {np.std(fds):.2f})")

Set: Real Test -> {'frechet': -5.4569682106375694e-12, 'precision': 1.0, 'recall': 1.0, 'nearest_train': 1.068061113357544}
Set: Copier -> {'frechet': 2.3599637933407394, 'precision': 0.9192200557103064, 'recall': 0.9220055710306406, 'nearest_train': 0.13467535376548767}
Set: Mean + Noise -> {'frechet': 138.18644722082573, 'precision': 0.9387186629526463, 'recall': 0.0, 'nearest_train': 1.6157054901123047}
Subset size 30: Mean FD = 21.18 (std 6.89)
Subset size 60: Mean FD = 11.32 (std 5.35)
Subset size 120: Mean FD = 5.77 (std 1.38)
Subset size 359: Mean FD = 2.81 (std 0.88)
Subset size 1000: Mean FD = 1.79 (std 0.22)
